# Mapeamento Eleitoral e Visualização Geospacial com Folium


In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import pathlib

In [6]:
data_path = pathlib.Path('./data')
sjdr_votacao_csv_path = data_path / 'votacao_sjdr.csv' 

In [7]:
df = pd.read_csv(sjdr_votacao_csv_path, sep=',')
df.head()

,DT_GERACAO_x,HH_GERACAO_x,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_PLEITO,DT_PLEITO,NR_TURNO_x,CD_ELEICAO,DS_ELEICAO_x,...,DS_SITU_LOCALIDADE,CD_SITU_SECAO_ACESSIBILIDADE,DS_SITU_SECAO_ACESSIBILIDADE,QT_ELEITOR_SECAO,QT_ELEITOR_ELEICAO_FEDERAL,QT_ELEITOR_ELEICAO_ESTADUAL,QT_ELEITOR_ELEICAO_MUNICIPAL,NR_LOCAL_VOTACAO_ORIGINAL,NM_LOCAL_VOTACAO_ORIGINAL,DS_ENDERECO_LOCVT_ORIGINAL
0,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,238,414,0,0,1023,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"
1,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,238,415,415,0,1023,COLEGIO NOSSA SENHORA DAS DORES,"RUA COMENDADOR BASTOS, 84"
2,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,323,325,325,0,1716,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,"PRAÇA BOM JESUS DE MATOZINHOS, S/N"
3,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,323,325,0,0,1716,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,"PRAÇA BOM JESUS DE MATOZINHOS, S/N"
4,2026-10-05,14:03:43,2026,0,Eleição Ordinária,3220,2026-10-04 00:00:00,1,6257,Eleição Geral Federal 2026,...,Ativo,0,Sem acessibilidade,359,356,356,0,1864,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,"AV. SETE DE SETEMBRO, S/N"


In [9]:
df_turno1 = df[df["NR_TURNO_x"] == 1]
df_turno2 = df[df["NR_TURNO_x"] == 2]

count_turno1 = len(df_turno1)
count_turno2 = len(df_turno2)

print("Qtd de registros turno 1:", count_turno1)
print("Qtd de registros turno 2:", count_turno2)

Qtd de registros turno 1: 3702
Qtd de registros turno 2: 1710


**Conta a quantidade de zonas eleitorais**

SJDR tem somente uma zona eleitoral (zona 328)

In [10]:
df_zona_1T = df_turno1.groupby('NR_ZONA')
df_zona_2T = df_turno2.groupby('NR_ZONA')

print(f'Zonas 1T: {len(df_zona_1T)}') 
print(f'Zonas 2T: {len(df_zona_2T)}')

Zonas 1T: 1
Zonas 2T: 1


**Conta a quantidade de secoes eleitorais**

A zona 328 tem 241 secoes eleitorais, contudo ela abrange cidades da regiao de SJDR, nao somente SJDR. Como filtramos apenas por SJDJ, e normal que tenham menos secoes que o total de secoes da zona.

In [11]:
secoes_1T = df_zona_1T['NR_SECAO'].nunique() 
secoes_2T = df_zona_2T['NR_SECAO'].nunique() 

print(f"Qtd secoes 1T: {secoes_1T.iloc[0]}")
print(f"Qtd secoes 2T: {secoes_2T.iloc[0]}")

Qtd secoes 1T: 218
Qtd secoes 2T: 218


## Limpeza, Filtragem e Unificação dos Dados

### Agrupa o Total de Votos por Candidato em Cada Seção em Ambos os Turnos

In [24]:
df_secoes_1T = (
    df_turno1.groupby(['NR_SECAO', 'NM_VOTAVEL'], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_SECAO', 'QT_VOTOS'], ascending=[True, False],)
)

df_secoes_1T.head(10)

,NR_SECAO,NM_VOTAVEL,QT_VOTOS
2,40,FLAVIO BOLSONARO,244
3,40,LULA,170
4,40,Nulo,18
6,40,RONALDO CAIADO,8
1,40,ESCRITOR AUGUSTO CURY,6
5,40,RENAN SANTOS,6
0,40,Branco,4
7,40,ZEMA,4
10,41,FLAVIO BOLSONARO,238
12,41,LULA,182


In [27]:
df_secoes_2T = (
    df_turno2.groupby(['NR_SECAO', 'NM_VOTAVEL'], as_index=False)['QT_VOTOS']
        .sum()
        .sort_values(by=['NR_SECAO', 'QT_VOTOS'], ascending=[True, False],)
)

df_secoes_2T.head(10)

,NR_SECAO,NM_VOTAVEL,QT_VOTOS
1,40,FLAVIO BOLSONARO,260
2,40,LULA,190
3,40,Nulo,12
0,40,Branco,2
5,41,FLAVIO BOLSONARO,272
6,41,LULA,218
7,41,Nulo,8
4,41,Branco,6
9,43,FLAVIO BOLSONARO,218
10,43,LULA,160


## Choropleth Maps do 1º Turno

**Transforma a latitude e longitude das zonas eleitorais em float**

In [30]:
df_turno1['NR_LATITUDE'] = pd.to_numeric(
    df_turno1['NR_LATITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)
df_turno2['NR_LATITUDE'] = pd.to_numeric(
    df_turno1['NR_LATITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)

df_turno1['NR_LONGITUDE'] = pd.to_numeric(
    df_turno1['NR_LONGITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)
df_turno2['NR_LONGITUDE'] = pd.to_numeric(
    df_turno1['NR_LONGITUDE']
    .astype(str)
    .str.strip() 
    .str.replace(',', '.')
)

**Obtem o mapa**

In [32]:
map = folium.Map(
    location=[-21.1365, -44.2619], # SJDR
    zoom_start=20,
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors' 
)

**Coloca marcadores nas secoes de votacao**

In [33]:
map_cols = [
    'NR_SECAO',
    'NR_LATITUDE',
    'NR_LONGITUDE'
]

df_lat_lon_secoes = df_turno1[map_cols].drop_duplicates()

for row in df_lat_lon_secoes.itertuples(index=False):
    lat = row.NR_LATITUDE
    lon = row.NR_LONGITUDE
    secao = row.NR_SECAO
    
    folium.Marker(
        location=[lat, lon],
        popup=f'Secao {secao}',
    ).add_to(map)

map